# 05 · Baseline classifier: TF-IDF + LinearSVC

The classical baseline for task A (7 classes). The fine-tuned DistilRoBERTa (`colab_gpu.ipynb`) is compared with it.

**Runs on a Colab CPU runtime**, not on the laptop. The TF-IDF matrix of the 116,417 train petitions has about
100 million non-zero values, and LinearSVC makes its own copy of it, so the peak memory need is about 3–4 GB.
The laptop usually has about 1 GB free; a Colab CPU runtime has 12.7 GB. No GPU is needed.

**Input**, in the Google Drive folder `DRIVE_DIR`: `petitions.parquet` and `splits.parquet` (already uploaded).
The code in `src/` comes from the GitHub repository.

**Output**, in `OUTPUT_DIR` on Google Drive:

| File | Content | Copy to (laptop) |
|---|---|---|
| `model.joblib` | the two TF-IDF vectorizers, the LinearSVC and the Platt calibrator | `models/tfidf_svc/` |
| `val_predictions.parquet` | `id`, `score_<class>`, `p_<class>` for the 4,030 validation petitions | `data/processed/predictions/tfidf_svc_val.parquet` |
| `c_tuning.csv` | validation macro-F1 for each C | `reports/tfidf_svc_c_tuning.csv` |
| `top_words.csv` | the words with the largest weight per class | `reports/tfidf_svc_top_words.csv` |
| `environment.json` | versions, settings, minutes, peak memory | `reports/tfidf_svc_environment.json` |

Steps: 1 settings → 2 Drive, code, libraries → 3 data → 4 TF-IDF features → 5 choose C → 6 Platt calibration →
7 validation results → 8 top words → 9 save. About 20–40 minutes. The test split is not used.

## 1. Settings

In [1]:
DRIVE_DIR = "/content/drive/MyDrive/capstone"    # Google Drive folder with the parquet files
OUTPUT_DIR = f"{DRIVE_DIR}/outputs/tfidf_svc"
REPO_URL = "https://github.com/YasithaNV2001/Petition-Moderation-Assistant-CS5998-Capstone-Project.git"
REPO_DIR = "/content/repo"
SKLEARN_VERSION = "1.9.1"    # the laptop's version (requirements.txt), so the saved model also loads there

# Locked settings (Milestone 1)
WORD_NGRAMS = (1, 2)         # single words and pairs of neighbouring words
CHAR_NGRAMS = (3, 5)         # pieces of 3 to 5 letters inside each word ("char_wb")
MIN_DF = 2                   # ignore words and pieces that appear in only one train petition
C_VALUES = [0.1, 0.3, 1, 3]  # tried on validation; the best macro-F1 wins
SEED = 42
TOP_N = 15                   # top words shown per class

## 2. Google Drive, code and libraries

Colab comes with an older scikit-learn. The model saved here is loaded again on the laptop (notebooks 08–09 and
the pipeline), and a model saved with one scikit-learn version may not load with another. So Colab first installs
the laptop's version.

In [2]:
%pip install -q scikit-learn==1.9.1

In [3]:
import json, os, platform, resource, subprocess, sys, time, warnings
import numpy as np
import pandas as pd
import scipy
import sklearn
from google.colab import drive

assert sklearn.__version__ == SKLEARN_VERSION, \
    f"scikit-learn {sklearn.__version__} is loaded: Runtime > Restart session, then run all cells again"

drive.mount("/content/drive")
for name in ["petitions.parquet", "splits.parquet"]:
    path = f"{DRIVE_DIR}/{name}"
    assert os.path.exists(path), f"Not found on Google Drive: {path} (check DRIVE_DIR in section 1)"
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                             capture_output=True, text=True, check=True).stdout.strip()
sys.path.insert(0, f"{REPO_DIR}/src")
from preprocess import CLASSES, MODEL_INPUTS, leakage_check

VERSIONS = {"python": platform.python_version(), "scikit-learn": sklearn.__version__, "scipy": scipy.__version__,
            "numpy": np.__version__, "pandas": pd.__version__}
TIMINGS = {}    # minutes per step
print("code from GitHub commit", REPO_COMMIT, "|", os.cpu_count(), "CPU cores")
for package, version in VERSIONS.items():
    print(f"  {package:14s} {version}")

Mounted at /content/drive
code from GitHub commit 6bb8e61 | 2 CPU cores
  python         3.13.18
  scikit-learn   1.9.1
  scipy          1.16.3
  numpy          2.1.3
  pandas         2.2.3


## 3. Load the data

Only `id`, `label` and `text` are read. The classes are numbered in the `CLASSES` order (published = 0, …,
local = 6), so column *i* of every output belongs to `CLASSES[i]`.

In [4]:
petitions = pd.read_parquet(f"{DRIVE_DIR}/petitions.parquet", columns=["id", "label", "text"])
splits = pd.read_parquet(f"{DRIVE_DIR}/splits.parquet")
petitions = petitions.merge(splits, on="id", how="left", validate="one_to_one")
assert len(petitions) == 127_191
assert petitions["split"].value_counts().to_dict() == {"train": 116_417, "test": 6_182, "val": 4_030, "excluded": 562}

label2id = {c: i for i, c in enumerate(CLASSES)}
ALL_LABELS = list(range(len(CLASSES)))
train = petitions[petitions["split"] == "train"]
val = petitions[petitions["split"] == "val"]
leakage_check(train[MODEL_INPUTS])    # the model sees only the text
leakage_check(val[MODEL_INPUTS])
y_train = train["label"].map(label2id).to_numpy()
y_val = val["label"].map(label2id).to_numpy()
print(f"train {len(train):,} | validation {len(val):,}")

train 116,417 | validation 4,030


## 4. TF-IDF features

Each petition becomes a long list of numbers, one per word or letter piece:

- **Word block:** single words and pairs of words ("tv licence"). Text is lower-cased.
- **Character block (`char_wb`):** pieces of 3–5 letters inside each word ("lic", "lice", "licen", …). They catch
  spelling variants, plurals and typos ("licence" / "license") that the word block treats as different words.
- **TF-IDF weight:** a word counts more when it is frequent in this petition (TF, damped with `sublinear_tf`:
  1 + log(count)) and rare across petitions (IDF). Each block is then scaled to length 1.
- `min_df=2`: words and pieces that appear in only one train petition are dropped.

The vocabulary and the IDF values are learned from the **train** petitions only; validation is only transformed.

In [5]:
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer

word_vectorizer = TfidfVectorizer(analyzer="word", ngram_range=WORD_NGRAMS, min_df=MIN_DF,
                                  sublinear_tf=True, lowercase=True)
char_vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=CHAR_NGRAMS, min_df=MIN_DF,
                                  sublinear_tf=True, lowercase=True)

start = time.time()
X_train_word = word_vectorizer.fit_transform(train["text"])
X_val_word = word_vectorizer.transform(val["text"])
X_train_char = char_vectorizer.fit_transform(train["text"])
X_val_char = char_vectorizer.transform(val["text"])
X_train = hstack([X_train_word, X_train_char], format="csr")    # word columns first, then char columns
X_val = hstack([X_val_word, X_val_char], format="csr")
N_WORD = X_train_word.shape[1]
del X_train_word, X_val_word, X_train_char, X_val_char           # only the joined matrices are needed now
TIMINGS["tfidf"] = round((time.time() - start) / 60, 1)

print(f"features: {N_WORD:,} word + {X_train.shape[1] - N_WORD:,} char = {X_train.shape[1]:,}")
print(f"train matrix {X_train.shape}: {X_train.nnz:,} non-zero values ({X_train.nnz / X_train.shape[0]:.0f} per petition), "
      f"{(X_train.data.nbytes + X_train.indices.nbytes) / 1e9:.2f} GB; {TIMINGS['tfidf']} min")

features: 678,715 word + 264,430 char = 943,145
train matrix (116417, 943145): 101,365,185 non-zero values (871 per petition), 1.22 GB; 2.6 min


## 5. LinearSVC: choose C on validation

A linear SVM learns one weight per feature and class; the class with the highest score wins (one-vs-rest).

- `class_weight="balanced"`: mistakes on rare classes (local, honours) cost more, as in DistilRoBERTa.
- **C** controls how closely the model fits the train petitions: a small C keeps the weights small and simple, a
  large C fits the train data more closely and risks overfitting. Each C is trained on train and scored on
  validation; the best **macro-F1** (the plain average of the 7 per-class F1 scores, so rare classes count as much
  as common ones) wins.
- "converged" tells whether the solver finished within its default 1,000 iterations.

In [6]:
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.svm import LinearSVC

results, models = [], {}
for C in C_VALUES:
    start = time.time()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model = LinearSVC(C=C, class_weight="balanced", random_state=SEED).fit(X_train, y_train)
    predicted = model.predict(X_val)
    results.append({"C": C,
                    "val_macro_f1": round(f1_score(y_val, predicted, average="macro", labels=ALL_LABELS), 4),
                    "val_accuracy": round(accuracy_score(y_val, predicted), 4),
                    "converged": not any(issubclass(w.category, ConvergenceWarning) for w in caught),
                    "minutes": round((time.time() - start) / 60, 1)})
    models[C] = model
    print(results[-1])

tuning = pd.DataFrame(results)
BEST_C = float(tuning.loc[tuning["val_macro_f1"].idxmax(), "C"])
svc = models[BEST_C]
assert list(svc.classes_) == ALL_LABELS
TIMINGS["svc_tuning"] = round(tuning["minutes"].sum(), 1)
tuning.to_csv(f"{OUTPUT_DIR}/c_tuning.csv", index=False)
print("best C:", BEST_C)
tuning

{'C': 0.1, 'val_macro_f1': 0.4452, 'val_accuracy': 0.6581, 'converged': True, 'minutes': 1.1}
{'C': 0.3, 'val_macro_f1': 0.4367, 'val_accuracy': 0.661, 'converged': True, 'minutes': 1.3}
{'C': 1, 'val_macro_f1': 0.4254, 'val_accuracy': 0.6583, 'converged': True, 'minutes': 2.5}
{'C': 3, 'val_macro_f1': 0.4085, 'val_accuracy': 0.6526, 'converged': True, 'minutes': 5.5}
best C: 0.1


,C,val_macro_f1,val_accuracy,converged,minutes
0,0.1,0.4452,0.6581,True,1.1
1,0.3,0.4367,0.6610,True,1.3
2,1.0,0.4254,0.6583,True,2.5
3,3.0,0.4085,0.6526,True,5.5


## 6. Platt calibration

The SVC gives **scores**, not probabilities: a score of 1.3 does not mean "73% sure". Platt calibration fits a
small S-shaped curve (sigmoid) per class on the **validation** petitions that turns each score into a probability;
the 7 values are then scaled to add up to 1. The SVC itself is frozen (`FrozenEstimator`), so it is not trained
again. The probabilities are used later for the confidence, the review flag and the fusion model (08).

Both are saved: the raw scores (`score_<class>`) and the probabilities (`p_<class>`), like the logits and
probabilities of DistilRoBERTa.

In [7]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

calibrator = CalibratedClassifierCV(FrozenEstimator(svc), method="sigmoid").fit(X_val, y_val)
assert len(calibrator.calibrated_classifiers_) == 1 and list(calibrator.classes_) == ALL_LABELS
val_scores = svc.decision_function(X_val)
val_probs = calibrator.predict_proba(X_val)

predictions = pd.DataFrame({"id": val["id"].to_numpy()})
for i, c in enumerate(CLASSES):
    predictions[f"score_{c}"] = val_scores[:, i]
for i, c in enumerate(CLASSES):
    predictions[f"p_{c}"] = val_probs[:, i]
predictions.to_parquet(f"{OUTPUT_DIR}/val_predictions.parquet", index=False)
assert np.allclose(val_probs.sum(axis=1), 1)
same = (val_probs.argmax(axis=1) == val_scores.argmax(axis=1)).mean()
print(f"{predictions.shape}; the highest probability and the highest score pick the same class for {same:.1%} of petitions")

(4030, 15); the highest probability and the highest score pick the same class for 82.0% of petitions


## 7. Validation results

The baseline's prediction is the class with the highest SVC score. C was chosen on these same validation petitions,
so the number is slightly optimistic (the same holds for DistilRoBERTa's best epoch); the fair comparison is on the
test split, once, in 09.

In [8]:
predicted = val_scores.argmax(axis=1)
VAL_MACRO_F1 = f1_score(y_val, predicted, average="macro", labels=ALL_LABELS)
print(f"validation macro-F1 {VAL_MACRO_F1:.4f}, accuracy {accuracy_score(y_val, predicted):.3f}\n")
print(classification_report(y_val, predicted, labels=ALL_LABELS, target_names=CLASSES, digits=3, zero_division=0))
pd.DataFrame(confusion_matrix(y_val, predicted, labels=ALL_LABELS),
             index=[f"true {c}" for c in CLASSES], columns=CLASSES)

validation macro-F1 0.4452, accuracy 0.658

                   precision    recall  f1-score   support

        published      0.725     0.951     0.823      1855
        duplicate      0.732     0.456     0.562      1161
       irrelevant      0.358     0.514     0.422       278
        no-action      0.444     0.367     0.402       455
          honours      0.636     0.833     0.722        42
already-happening      0.188     0.066     0.098        91
            local      0.700     0.047     0.089       148

         accuracy                          0.658      4030
        macro avg      0.540     0.462     0.445      4030
     weighted avg      0.656     0.658     0.628      4030



,published,duplicate,irrelevant,no-action,honours,already-happening,local
true published,1765,48,20,18,0,4,0
true duplicate,381,529,81,142,14,13,1
true irrelevant,74,28,143,27,1,5,0
true no-action,146,87,44,167,5,4,2
true honours,1,5,1,0,35,0,0
true already-happening,39,24,4,18,0,6,0
true local,29,2,106,4,0,0,7


## 8. Top words per class

The words (and word pairs) with the largest weights for each class: they push a petition towards that class.
Only the word block is shown, because letter pieces are hard to read.

In [9]:
word_names = word_vectorizer.get_feature_names_out()
top_words = pd.DataFrame({c: word_names[np.argsort(svc.coef_[i, :N_WORD])[::-1][:TOP_N]]
                          for i, c in enumerate(CLASSES)})
top_words.to_csv(f"{OUTPUT_DIR}/top_words.csv", index=False)
top_words

,published,duplicate,irrelevant,no-action,honours,already-happening,local
0,we believe,general election,council,petitions,sack,super league,council
1,we think,second referendum,mp,the way,no confidence,80,more ambulances
2,government should,election,resign,sex education,he,self employed,train station
3,require,bank holiday,pay cut,as disability,his,garden centres,station
4,fund,state visit,expenses,was not,honours,super,ambulances to
5,believe,sign language,mp pay,parking charges,knighthood,thomas moore,post office
6,could help,without deal,in northern,900 000,peerage,european super,north east
7,fund free,all nhs,in wales,plane,she,holiday parks,planning there
8,provide funding,recess,uber,support to,remove,tom knighthood,rifle
9,the government,close schools,in scotland,school system,sir,baby classes,help fund


## 9. Save the model and the record

One file holds everything needed to classify a new petition: the two vectorizers, the SVC and the calibrator. It is
loaded again here, as a check that the saved file gives the same scores.

In [10]:
import joblib

model_path = f"{OUTPUT_DIR}/model.joblib"
joblib.dump({"word_vectorizer": word_vectorizer, "char_vectorizer": char_vectorizer, "svc": svc,
             "calibrator": calibrator, "classes": CLASSES}, model_path)

saved = joblib.load(model_path)    # how 09 and the pipeline will use it
texts = val["text"].iloc[:200]
X_check = hstack([saved["word_vectorizer"].transform(texts), saved["char_vectorizer"].transform(texts)], format="csr")
assert np.allclose(saved["svc"].decision_function(X_check), val_scores[:200])
assert np.allclose(saved["calibrator"].predict_proba(X_check), val_probs[:200])
print(f"model.joblib: {os.path.getsize(model_path) / 1e6:.0f} MB, loads and gives the same scores")

model.joblib: 77 MB, loads and gives the same scores


In [11]:
environment = {
    "notebook": "05_baseline_classifier",
    "date_utc": pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds"),
    "runtime": f"Colab CPU, {os.cpu_count()} cores",
    "peak_memory_gb": round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, 1),   # Linux reports kB
    "versions": VERSIONS,
    "code_commit": REPO_COMMIT,
    "settings": {"word_ngrams": WORD_NGRAMS, "char_ngrams": CHAR_NGRAMS, "min_df": MIN_DF, "sublinear_tf": True,
                 "class_weight": "balanced", "C_values": C_VALUES, "calibration": "Platt (sigmoid) on validation",
                 "seed": SEED},
    "features": {"word": N_WORD, "char": X_train.shape[1] - N_WORD, "train_nonzeros": int(X_train.nnz)},
    "c_tuning": results,
    "best_C": BEST_C,
    "val_macro_f1": round(float(VAL_MACRO_F1), 4),
    "minutes": TIMINGS,
}
with open(f"{OUTPUT_DIR}/environment.json", "w") as f:
    json.dump(environment, f, indent=2)
print(json.dumps(environment, indent=2))

{
  "notebook": "05_baseline_classifier",
  "date_utc": "2026-11-02T12:31:07+00:00",
  "runtime": "Colab CPU, 2 cores",
  "peak_memory_gb": 4.3,
  "versions": {
    "python": "3.13.18",
    "scikit-learn": "1.9.1",
    "scipy": "1.16.3",
    "numpy": "2.1.3",
    "pandas": "2.2.3"
  },
  "code_commit": "6bb8e61",
  "settings": {
    "word_ngrams": [
      1,
      2
    ],
    "char_ngrams": [
      3,
      5
    ],
    "min_df": 2,
    "sublinear_tf": true,
    "class_weight": "balanced",
    "C_values": [
      0.1,
      0.3,
      1,
      3
    ],
    "calibration": "Platt (sigmoid) on validation",
    "seed": 42
  },
  "features": {
    "word": 678715,
    "char": 264430,
    "train_nonzeros": 101365185
  },
  "c_tuning": [
    {
      "C": 0.1,
      "val_macro_f1": 0.4452,
      "val_accuracy": 0.6581,
      "converged": true,
      "minutes": 1.1
    },
    {
      "C": 0.3,
      "val_macro_f1": 0.4367,
      "val_accuracy": 0.661,
      "converged": true,
      "minutes": 1

## Files to download

Download the `tfidf_svc` folder from Drive (`outputs/tfidf_svc`) and copy the files as shown in the table at the
top. Also save this notebook with its outputs (File → Download → .ipynb) as `notebooks/05_baseline_classifier.ipynb`.

In [12]:
for name in sorted(os.listdir(OUTPUT_DIR)):
    print(f"{name:28s} {os.path.getsize(f'{OUTPUT_DIR}/{name}') / 1e6:8.1f} MB")

c_tuning.csv                      0.0 MB
environment.json                  0.0 MB
model.joblib                     77.0 MB
top_words.csv                     0.0 MB
val_predictions.parquet           0.6 MB
